In [1]:
# import all necessary libraries
import h5py
import numpy as np
import vaex
import pandas as pd
from pathlib import Path as path

In [2]:
# define galaxy info
box = "storm"
gal_num = 2
galaxy_name = f'{box}_{gal_num}'

distances = [2000, 4000]
depths = [25.0, "26p5.0"]
combo_1 = [distances[0], depths[0]]
combo_2 = [distances[0], depths[1]]
combo_3 = [distances[1], depths[0]]
combo_4 = [distances[1], depths[1]]
combos = [combo_1, combo_2, combo_3, combo_4]

In [3]:
num_angles = 3
decs = [1, 2, 3]
azis = [4, 5, 6]

In [4]:
def angle_view(pos_array, dec, azi):
    # spherical coordinates. Turns into radians.
    theta = dec
    phi = azi
    dec_readable = round(dec*180/np.pi,1)
    azi_readable = round(np.mod(azi*180/np.pi, 360),1)

    # use rotation matrix to find 2d projected image from any given angle of observation
    project_matrix = np.array([[-np.sin(phi),                np.cos(phi),             0            ],
                             [-np.cos(theta)*np.cos(phi), -np.cos(theta)*np.sin(phi), np.sin(theta)]])

    project_pos_array_all = np.matmul(project_matrix, pos_array) #applies 2x3 transformation to 3xn data. Result is 2xn data 

    return dec_readable, azi_readable, project_pos_array_all

In [5]:
for i in range(4):
    for j in range(len(decs)):
        distance = combos[i][0]
        depth = combos[i][1]
    
        dwarfcatpath = f'raw_data/{galaxy_name}/survey.{box}_4096_2_data_{distance}_{depth}.h5'
        vdwarfcat = vaex.open(dwarfcatpath)
        dwarfcat = pd.DataFrame(vdwarfcat,columns=vdwarfcat.column_names)
        
        size_kpc = 40 
        pdist = (size_kpc/distance) * (180/np.pi) 
        year = 10
        mlim = '25'
        c1='px'
        c2='py'
        edgelength = 10
    
        # adjust ananke STAR data so that center of mass is the origin.
        ananke_data = np.column_stack([dwarfcat['px'],dwarfcat['py'], dwarfcat['pz']])
        ananke_masses = dwarfcat['smass']
        ananke_total_mass = np.sum(ananke_masses)
        
        cm_current_ananke = [0,0,0]
        for k in range(len(ananke_data)):
            cm_current_ananke = cm_current_ananke + ananke_data[i] * ananke_masses[i]
        cm_ananke = [x/ananke_total_mass for x in cm_current_ananke]
        
        ananke_data_centered = ananke_data - cm_ananke

        d, a, altered_positions = angle_view(ananke_data_centered.T, decs[j], azis[j])
        dwarfcat['px'] = altered_positions.T[:,0]
        dwarfcat['py'] = altered_positions.T[:,1]

        file_path = path(f'rotated_catalogs/{galaxy_name}/{distance}_{depth}/{galaxy_name}_d={decs[j]}_a={azis[j]}.h5')
        file_path.parent.mkdir(parents=True, exist_ok=True)

        vframe_new = vaex.from_pandas(dwarfcat)
        vframe_new.export_hdf5(file_path, progress=True)

export(hdf5) [########################################] 100.00% elapsed time  :     0.14s =  0.0m =  0.0h
export(hdf5) [########################################] 100.00% elapsed time  :     0.12s =  0.0m =  0.0h
export(hdf5) [########################################] 100.00% elapsed time  :     0.12s =  0.0m =  0.0h
export(hdf5) [########################################] 100.00% elapsed time  :     0.50s =  0.0m =  0.0h
export(hdf5) [########################################] 100.00% elapsed time  :     0.48s =  0.0m =  0.0h
export(hdf5) [########################################] 100.00% elapsed time  :     0.44s =  0.0m =  0.0h
export(hdf5) [########################################] 100.00% elapsed time  :     0.10s =  0.0m =  0.0h
export(hdf5) [########################################] 100.00% elapsed time  :     0.07s =  0.0m =  0.0h
export(hdf5) [########################################] 100.00% elapsed time  :     0.07s =  0.0m =  0.0h
export(hdf5) [################################